# nbplay host smoke test

Rendered by Voila and JupyterLab in the browser test suite: every session widget must render and stay in sync through a live kernel.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from nbplay import SequencerWidget, Session, SynthWidget

session = Session(bpm=120.0)
seq = SequencerWidget(length=8)
for i in range(0, 8, 2):
    seq.set_step(i, note=60 + i, active=True)
session.add_track("Lead", seq, SynthWidget())
session.launcher.add_scene("A")
session.launcher.set_slot(0, 0, seq, name="Lead A")

# Kernel-side probe: proves browser -> kernel sync works under this host.
state = widgets.HTML(value='<span id="nbplay-kernel-state">kernel: stopped</span>')


def on_play(change):
    state.value = f'<span id="nbplay-kernel-state">kernel: {"playing" if change["new"] else "stopped"}</span>'


session.transport.observe(on_play, names="is_playing")

# Kernel -> browser: a plain ipywidgets button that stops the session from Python.
stop_button = widgets.Button(description="Stop from kernel")
stop_button.add_class("nbplay-voila-stop")
stop_button.on_click(lambda _: session.stop())

display(
    widgets.VBox(
        [
            state,
            stop_button,
            session.transport,
            seq,
            session.launcher,
            session.timeline,
            session.mixer,
        ]
    )
)
